In [42]:
# %% Cell 1 — Imports & setup
import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR  = Path("../public")
OUT_DIR.mkdir(exist_ok=True)

XLSX =  "Feb26CBOProjections.xlsx"

xl = pd.ExcelFile(XLSX)
print("Sheets:", xl.sheet_names)

# %% Cell 2 — Helpers
def clean(v):
    if isinstance(v, float) and np.isnan(v):
        return None
    s = str(v).strip().strip("'").strip()
    return s if s not in ("", "*", "—", "n.a.", " ") else None

def to_num(v):
    c = clean(v)
    if c is None:
        return None
    try:
        return float(str(c).replace(",", ""))
    except:
        return None

def is_year(v):
    c = clean(v)
    if c is None:
        return False
    try:
        y = int(float(c))
        return 2024 <= y <= 2037
    except:
        return False

# %% Cell 3 — Raw inspection helper
def peek(sheet, rows=range(0, 25)):
    raw = pd.read_excel(XLSX, sheet_name=sheet, header=None)
    print(f"=== {sheet} ({len(raw)} rows x {len(raw.columns)} cols) ===")
    for i in rows:
        if i >= len(raw):
            break
        row = raw.iloc[i]
        vals = [repr(v) for v in row if not (isinstance(v, float) and np.isnan(v))]
        if vals:
            print(f"  row {i:2d}: {vals}")

Sheets: ['Contents', 'Table 1-1', 'Table 1-2', 'Table 1-3', 'Table 1-4', 'Table 3-1', 'Table 3-2', 'Table 3-2, unadj', 'Table 3-3', 'Table 3-4', 'Table 3-5', 'Table 3-6', 'Table 3-7', 'Table 3-8', 'Box 3-2 Table', 'Table 5-1']


In [43]:
# %% Cell 4 — Parse Table 1-1 by explicit row index
# Structure (confirmed from peek):
#   row  8: year headers (cols 1-12 = 2025-2036, cols 13-14 = summary totals → skip)
#   row 16: Total Revenues
#   row 20: Mandatory Outlays
#   row 21: Discretionary Outlays
#   row 22: Net Interest
#   row 23: Total Outlays
#   rows 11-15: individual revenue line items
# "On-budget"/"Off-budget" rows skipped (not needed)

def parse_table_1_1(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 1-1", header=None)

    # Build col -> year mapping from row 8, skipping summary cols
    col_years = {}
    for ci, v in enumerate(raw.iloc[8]):
        if is_year(v):
            col_years[ci] = int(float(clean(v)))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Individual Income Taxes": get_row(11),
        "Payroll Taxes":           get_row(12),
        "Corporate Income Taxes":  get_row(13),
        "Customs Duties":          get_row(14),
        "Other Revenues":          get_row(15),
        "Total Revenues":          get_row(16),
        "Mandatory Outlays":       get_row(20),
        "Discretionary Outlays":   get_row(21),
        "Net Interest":            get_row(22),
        "Total Outlays":           get_row(23),
        "Total Deficit":           get_row(26),   # negative in source — abs() when using
        "Debt Held by Public":     get_row(30),
        "GDP":                     get_row(32),   # Addendum row, past "Addendum:" header
    }

    return sorted(col_years.values()), data

years_11, data_11 = parse_table_1_1(XLSX)

print(f"Years: {years_11}")
print("\nSpot checks (2025 / 2026):")
checks = {
    "Total Revenues":    (5234.6,   5595.9),
    "Total Outlays":     (7009.986, 7448.619),
    "Net Interest":      (969.938,  1038.976),
    "Mandatory Outlays": (4167.623, 4529.339),
    "Total Deficit":     (-1775.37, -1852.703),  # negative
    "Debt Held by Public":(30172.402,32095.165),
    "GDP":               (30362.025,31902.006),
}
for label, (exp25, exp26) in checks.items():
    v25 = data_11[label][2025]
    v26 = data_11[label][2026]
    ok25 = "✓" if abs(v25 - exp25) < 0.1 else "✗"
    ok26 = "✓" if abs(v26 - exp26) < 0.1 else "✗"
    print(f"  {label:25s} 2025: {v25:.1f} {ok25}  2026: {v26:.1f} {ok26}")


Years: [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036]

Spot checks (2025 / 2026):
  Total Revenues            2025: 5234.6 ✓  2026: 5595.9 ✓
  Total Outlays             2025: 7010.0 ✓  2026: 7448.6 ✓
  Net Interest              2025: 969.9 ✓  2026: 1039.0 ✓
  Mandatory Outlays         2025: 4167.6 ✓  2026: 4529.3 ✓
  Total Deficit             2025: -1775.4 ✓  2026: -1852.7 ✓
  Debt Held by Public       2025: 30172.4 ✓  2026: 32095.2 ✓
  GDP                       2025: 30362.0 ✓  2026: 31902.0 ✓


In [44]:
peek("Table 1-3")

=== Table 1-3 (39 rows x 13 cols) ===
  row  0: ["'This file presents data that supplement CBO’s February 2026 report The Budget and Economic Outlook: 2026 to 2036.'"]
  row  1: ["'www.cbo.gov/publication/61882'"]
  row  4: ['"Table 1-3. \\nCBO\'s Baseline Projections of Federal Debt"']
  row  5: ["'Billions of dollars'"]
  row  7: ["'Actual, \\n2025'", 'np.float64(2026.0)', 'np.float64(2027.0)', 'np.float64(2028.0)', 'np.float64(2029.0)', 'np.float64(2030.0)', 'np.float64(2031.0)', 'np.float64(2032.0)', 'np.float64(2033.0)', 'np.float64(2034.0)', 'np.float64(2035.0)', 'np.float64(2036.0)']
  row  8: ["'Debt held by the public at the \\nbeginning of the year'", '28195.575', 'np.float64(30172.402)', 'np.float64(32095.165)', 'np.float64(34004.52)', 'np.float64(36092.745)', 'np.float64(38102.757)', 'np.float64(40279.639)', 'np.float64(42528.336)', 'np.float64(44922.49)', 'np.float64(47644.187)', 'np.float64(50393.853)', 'np.float64(53103.223)']
  row 10: ["'Changes in debt held by the pub

In [45]:
# %% Cell 5 — Parse Table 1-3 by explicit row index
def parse_table_1_3(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 1-3", header=None)

    # Row 7: year headers — first col is "Actual, \n2025" (not a bare year), rest are floats
    col_years = {}
    for ci, v in enumerate(raw.iloc[7]):
        c = clean(v)
        if c is None:
            continue
        # Handle "Actual, \n2025" specially
        if "2025" in c:
            col_years[ci] = 2025
            continue
        if is_year(v):
            col_years[ci] = int(float(c))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Debt at Beginning of Year":     get_row(8),
        "Change from Deficit":           get_row(11),
        "Change from Other Financing":   get_row(12),
        "Debt at End of Year (Billions)": get_row(16),
        "Debt at End of Year (% GDP)":   get_row(17),
        "Federal Financial Assets":      get_row(20),
    }

    return sorted(col_years.values()), data

years_13, data_13 = parse_table_1_3(XLSX)

print(f"Years: {years_13}")
print("\nSpot checks (2025 / 2026):")
checks_13 = {
    "Debt at Beginning of Year":      (28195.575, 30172.402),
    "Change from Deficit":            (1775.357,  1852.703),
    "Debt at End of Year (Billions)": (30172.402, 32095.165),
    "Debt at End of Year (% GDP)":    (99.375,    100.605),
}
for label, (exp25, exp26) in checks_13.items():
    v25 = data_13[label][2025]
    v26 = data_13[label][2026]
    ok25 = "✓" if abs(v25 - exp25) < 0.01 else "✗"
    ok26 = "✓" if abs(v26 - exp26) < 0.01 else "✗"
    print(f"  {label:35s} 2025: {v25:.3f} {ok25}  2026: {v26:.3f} {ok26}")


Years: [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036]

Spot checks (2025 / 2026):
  Debt at Beginning of Year           2025: 28195.575 ✓  2026: 30172.402 ✓
  Change from Deficit                 2025: 1775.357 ✓  2026: 1852.703 ✓
  Debt at End of Year (Billions)      2025: 30172.402 ✓  2026: 32095.165 ✓
  Debt at End of Year (% GDP)         2025: 99.375 ✓  2026: 100.605 ✓


In [46]:
# %% Cell 6 — Parse Table 5-1 by explicit row index
def parse_table_5_1(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 5-1", header=None)

    # Row 7: year headers 2026-2035 only (no 2025, no 2036); skip summary cols
    col_years = {}
    for ci, v in enumerate(raw.iloc[7]):
        if is_year(v):
            col_years[ci] = int(float(clean(v)))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Jan 2025 Baseline Deficit":         get_row(9),    # pre-OBBBA
        "Feb 2026 Baseline Deficit":         get_row(113),  # current law
        "Net Interest Change Legislative":   get_row(49),
        "Net Interest Change Economic":      get_row(75),
        "Net Interest Change Technical":     get_row(107),
    }

    return sorted(col_years.values()), data

years_51, data_51 = parse_table_5_1(XLSX)

print(f"Years: {years_51}")
print("\nSpot checks:")
checks_51 = {
    "Jan 2025 Baseline Deficit":       {2026: 1713.258, 2027: 1686.942, 2035: 2531.378},
    "Feb 2026 Baseline Deficit":       {2026: 1852.703, 2027: 1887.203, 2035: 2779.034},
    "Net Interest Change Legislative": {2026: 4.696,    2027: 27.514,   2035: 121.622},
    "Net Interest Change Economic":    {2026: 14.543,   2027: 11.248,   2035: 210.593},
    "Net Interest Change Technical":   {2026: 9.461,    2027: -6.285,   2035: -95.728},
}
for label, expected in checks_51.items():
    print(f"  {label}:")
    for yr, exp in expected.items():
        v = data_51[label][yr]
        ok = "✓" if abs(v - exp) < 0.01 else "✗"
        print(f"    {yr}: {v:.3f}  (expect {exp}) {ok}")



Years: [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035]

Spot checks:
  Jan 2025 Baseline Deficit:
    2026: 1713.258  (expect 1713.258) ✓
    2027: 1686.942  (expect 1686.942) ✓
    2035: 2531.378  (expect 2531.378) ✓
  Feb 2026 Baseline Deficit:
    2026: 1852.703  (expect 1852.703) ✓
    2027: 1887.203  (expect 1887.203) ✓
    2035: 2779.034  (expect 2779.034) ✓
  Net Interest Change Legislative:
    2026: 4.696  (expect 4.696) ✓
    2027: 27.514  (expect 27.514) ✓
    2035: 121.622  (expect 121.622) ✓
  Net Interest Change Economic:
    2026: 14.543  (expect 14.543) ✓
    2027: 11.248  (expect 11.248) ✓
    2035: 210.593  (expect 210.593) ✓
  Net Interest Change Technical:
    2026: 9.461  (expect 9.461) ✓
    2027: -6.285  (expect -6.285) ✓
    2035: -95.728  (expect -95.728) ✓


In [47]:
# %% Cell 7 — Build projections_deficit.csv (scenarios A & B)
rows = []

# Scenario A: Jan 2025 baseline (pre-OBBBA), years 2026-2035
for yr, v in sorted(data_51["Jan 2025 Baseline Deficit"].items()):
    if v is not None:
        rows.append({"scenario": "jan_2025_baseline", "year": yr, "deficit_billions": round(v, 1)})

# Scenario B: Feb 2026 current law (from Table 1-1, abs value, years 2025-2036)
for yr in sorted(years_11):
    v = data_11["Total Deficit"].get(yr)
    if v is not None:
        rows.append({"scenario": "feb_2026_current_law", "year": yr, "deficit_billions": round(abs(v), 1)})

df_deficit = pd.DataFrame(rows).sort_values(["scenario", "year"])

print(df_deficit.pivot(index="year", columns="scenario", values="deficit_billions").to_string())

print("\n10-year totals (2026-2035):")
for sc in ["jan_2025_baseline", "feb_2026_current_law"]:
    total = df_deficit[(df_deficit.scenario == sc) & df_deficit.year.between(2026, 2035)]["deficit_billions"].sum()
    print(f"  {sc:25s}: ${total:,.0f}B")
print("  Expected: jan ~$21,758B  |  feb ~$23,143B")

scenario  feb_2026_current_law  jan_2025_baseline
year                                             
2025                    1775.4                NaN
2026                    1852.7             1713.3
2027                    1887.2             1686.9
2028                    2079.7             1910.8
2029                    2019.7             1938.3
2030                    2200.6             2139.7
2031                    2285.7             2232.8
2032                    2439.4             2370.6
2033                    2780.7             2637.3
2034                    2818.5             2597.2
2035                    2779.0             2531.4
2036                    3115.4                NaN

10-year totals (2026-2035):
  jan_2025_baseline        : $21,758B
  feb_2026_current_law     : $23,143B
  Expected: jan ~$21,758B  |  feb ~$23,143B


In [ ]:
# %% Cell 8 — Derive all net interest scenarios
#
# Three scenarios, all grounded directly in CBO Table 1-1 and Table 5-1:
#
#   feb_2026_current_law  : Table 1-1 row 22 directly
#   jan_2025_baseline     : feb minus the sum of all three NI change components
#                           (legislative row 49 + economic row 75 + technical row 107)
#                           Validated: sum matches Table 5-1 Addendum row to the dollar.
#   no_tariff_revenue     : feb minus the TECHNICAL NI change only (row 107)
#                           Rationale: the technical NI change is dominated by interest
#                           savings from tariff revenues reducing debt accumulation.
#                           Removing it gives the NI path if those tariffs never materialised.
#
# No dependency on df_deficit_full — uses only data_11 and data_51 from Cells 4 & 6.

rows_ni = []

# ── Feb 2026 — directly from Table 1-1 row 22 ────────────────────────────────
for yr in sorted(years_11):
    v = data_11["Net Interest"].get(yr)
    if v is not None:
        rows_ni.append({
            "scenario": "feb_2026_current_law",
            "year": yr,
            "net_interest_billions": round(v, 3),
        })

# ── Jan 2025 — back out all three NI change components ───────────────────────
jan_ni_dict = {}
for yr in sorted(years_51):
    feb  = data_11["Net Interest"].get(yr)
    leg  = data_51["Net Interest Change Legislative"].get(yr)
    econ = data_51["Net Interest Change Economic"].get(yr)
    tech = data_51["Net Interest Change Technical"].get(yr)
    if all(v is not None for v in [feb, leg, econ, tech]):
        jan = feb - leg - econ - tech
        jan_ni_dict[yr] = jan
        rows_ni.append({
            "scenario": "jan_2025_baseline",
            "year": yr,
            "net_interest_billions": round(jan, 3),
        })

# ── No-tariff — remove the technical NI component only ───────────────────────
# Formula: no_tar_ni[yr] = feb_ni[yr] - tech_ni[yr]
# Where tech_ni is the interest savings CBO attributes to tariff-driven debt reduction.
# Removing it gives the NI path in a world without those tariff revenues.
# Note: tech_ni[2026] = +9.5 (slightly raises NI), tech_ni[2027:2035] = negative
# (tariff revenues reduce debt -> lower interest), so no_tar_ni > feb_ni from 2027 on.
for yr in sorted(years_51):
    feb  = data_11["Net Interest"].get(yr)
    tech = data_51["Net Interest Change Technical"].get(yr)
    if feb is not None and tech is not None:
        no_tar = feb - tech
        rows_ni.append({
            "scenario": "no_tariff_revenue",
            "year": yr,
            "net_interest_billions": round(no_tar, 3),
        })

df_ni = pd.DataFrame(rows_ni).sort_values(["scenario", "year"])

# ── Validation ────────────────────────────────────────────────────────────────
print("=== NET INTEREST BY SCENARIO (billions) ===")
pivot = df_ni[df_ni.year.between(2026, 2035)].pivot(
    index="year", columns="scenario", values="net_interest_billions"
)
# Reorder columns for readability
col_order = [c for c in ["jan_2025_baseline", "feb_2026_current_law", "no_tariff_revenue"] if c in pivot.columns]
print(pivot[col_order].to_string())

print("\n10yr totals (2026-2035):")
for sc in col_order:
    total = df_ni[(df_ni["scenario"] == sc) & df_ni["year"].between(2026, 2035)]["net_interest_billions"].sum()
    print(f"  {sc:35s}: ${total/1000:.2f}T")

print("\nMonotonicity checks (each series should increase year over year):")
for sc in col_order:
    vals = df_ni[(df_ni["scenario"] == sc) & df_ni["year"].between(2026, 2035)].sort_values("year")["net_interest_billions"].tolist()
    mono = all(vals[i] <= vals[i+1] for i in range(len(vals)-1))
    print(f"  {sc:35s}: {'✓ monotonic' if mono else '✗ NOT monotonic — check pipeline'}")

print("\nOrdering checks (jan < feb < no_tar for most years):")
for yr in range(2026, 2036):
    row = {sc: df_ni[(df_ni["scenario"]==sc) & (df_ni["year"]==yr)]["net_interest_billions"].values for sc in col_order}
    if all(len(v) for v in row.values()):
        jan_v, feb_v, not_v = row["jan_2025_baseline"][0], row["feb_2026_current_law"][0], row["no_tariff_revenue"][0]
        ok = "✓" if jan_v < feb_v and feb_v < not_v else ("~" if jan_v < not_v else "✗")
        print(f"  {yr}: jan={jan_v:.1f}  feb={feb_v:.1f}  no_tar={not_v:.1f}  {ok}")

scenario  feb_2026_current_law  jan_2025_baseline  no_tariff_revenue
year                                                                
2025                   969.938                NaN                NaN
2026                  1038.976           1010.276           1299.920
2027                  1107.715           1075.238           1427.595
2028                  1217.824           1164.485           1484.798
2029                  1324.496           1247.230           1527.781
2030                  1431.836           1327.549           1583.293
2031                  1548.383           1416.982           1670.069
2032                  1670.447           1513.903           1771.201
2033                  1784.364           1604.542           1892.864
2034                  1903.751           1693.559           2053.633
2035                  2019.072           1782.585           2196.719
2036                  2144.336                NaN                NaN

10yr totals (2026-2035):
  feb_20

In [49]:
# %% Cell 9 — Write output CSVs

# 1. projections_deficit.csv
df_deficit.to_csv(OUT_DIR / "projections_deficit.csv", index=False)
print(f"Wrote projections_deficit.csv ({len(df_deficit)} rows)")

# 2. projections_net_interest.csv
df_ni.to_csv(OUT_DIR / "projections_net_interest.csv", index=False)
print(f"Wrote projections_net_interest.csv ({len(df_ni)} rows)")

# 3. projections_summary.csv — key annual series from Table 1-1 and 1-3
summary_rows = []
cats_11 = {
    "Total Revenues":        "Total Revenues",
    "Total Outlays":         "Total Outlays",
    "Mandatory Outlays":     "Mandatory Outlays",
    "Discretionary Outlays": "Discretionary Outlays",
    "Net Interest":          "Net Interest",
    "Debt Held by Public":   "Debt Held by Public",
    "GDP":                   "GDP",
}
for label, key in cats_11.items():
    for yr in sorted(years_11):
        v = data_11[key].get(yr)
        if v is not None:
            summary_rows.append({"category": label, "year": yr, "amount_billions": round(v, 1)})

# Add debt % GDP from Table 1-3
for yr in sorted(years_13):
    v = data_13["Debt at End of Year (% GDP)"].get(yr)
    if v is not None:
        summary_rows.append({"category": "Debt Held by Public (% GDP)", "year": yr, "amount_billions": round(v, 3)})

df_summary = pd.DataFrame(summary_rows).sort_values(["category", "year"])
df_summary.to_csv(OUT_DIR / "projections_summary.csv", index=False)
print(f"Wrote projections_summary.csv ({len(df_summary)} rows)")

print("\nAll done. Files in:", OUT_DIR.resolve())

rows = []

# Scenario A: Jan 2025 baseline (pre-OBBBA), years 2026-2035
for yr, v in sorted(data_51["Jan 2025 Baseline Deficit"].items()):
    if v is not None:
        rows.append({"scenario": "jan_2025_baseline", "year": yr, "deficit_billions": round(v, 1)})

# Scenario B: Feb 2026 current law (from Table 1-1, abs value, years 2025-2036)
for yr in sorted(years_11):
    v = data_11["Total Deficit"].get(yr)
    if v is not None:
        rows.append({"scenario": "feb_2026_current_law", "year": yr, "deficit_billions": round(abs(v), 1)})

df_deficit = pd.DataFrame(rows).sort_values(["scenario", "year"])

print(df_deficit.pivot(index="year", columns="scenario", values="deficit_billions").to_string())

print("\n10-year totals (2026-2035):")
for sc in ["jan_2025_baseline", "feb_2026_current_law"]:
    total = df_deficit[(df_deficit.scenario == sc) & df_deficit.year.between(2026, 2035)]["deficit_billions"].sum()
    print(f"  {sc:25s}: ${total:,.0f}B")
print("  Expected: jan ~$21,758B  |  feb ~$23,143B")

Wrote projections_deficit.csv (22 rows)
Wrote projections_net_interest.csv (32 rows)
Wrote projections_summary.csv (96 rows)

All done. Files in: /Users/aaditbhatia/Desktop/OBBB Dashboard/viz/public
scenario  feb_2026_current_law  jan_2025_baseline
year                                             
2025                    1775.4                NaN
2026                    1852.7             1713.3
2027                    1887.2             1686.9
2028                    2079.7             1910.8
2029                    2019.7             1938.3
2030                    2200.6             2139.7
2031                    2285.7             2232.8
2032                    2439.4             2370.6
2033                    2780.7             2637.3
2034                    2818.5             2597.2
2035                    2779.0             2531.4
2036                    3115.4                NaN

10-year totals (2026-2035):
  jan_2025_baseline        : $21,758B
  feb_2026_current_law     : $23,1

In [50]:
# %% Cell 10 — Add no-tariff scenario to projections_deficit.csv
# "No tariff" = Feb 2026 baseline + customs duties technical change (Table 5-1 row 84)
# Row 84 is the tariff revenue that CBO assumes materializes — removing it shows
# what happens if tariffs are struck down or don't materialize.

tariff_technical = data_51  # already parsed — need row 84 separately
# Parse row 84 directly
raw_51 = pd.read_excel(XLSX, sheet_name="Table 5-1", header=None)

# Re-use col_years from parse_table_5_1
col_years_51 = {}
for ci, v in enumerate(raw_51.iloc[7]):
    if is_year(v):
        col_years_51[ci] = int(float(clean(v)))

row84 = raw_51.iloc[84]
tariff_rev_change = {yr: to_num(row84.iloc[ci]) for ci, yr in col_years_51.items()}

print("Tariff technical change by year (billions):")
for yr, v in sorted(tariff_rev_change.items()):
    print(f"  {yr}: {v}")

# Build no-tariff rows: feb_2026 deficit + tariff revenue change
# tariff_rev_change is positive (revenue increase) so adding it back increases the deficit
no_tariff_rows = []
feb_dict = {r["year"]: r["deficit_billions"]
            for r in df_deficit.to_dict("records")
            if r["scenario"] == "feb_2026_current_law"}

for yr, tariff in sorted(tariff_rev_change.items()):
    base = feb_dict.get(yr)
    if base is not None and tariff is not None:
        no_tariff_rows.append({
            "scenario": "no_tariff_revenue",
            "year": yr,
            "deficit_billions": round(base + tariff, 1)
        })

# Append to df_deficit and re-save
df_no_tariff = pd.DataFrame(no_tariff_rows)
df_deficit_full = pd.concat([df_deficit, df_no_tariff], ignore_index=True).sort_values(["scenario", "year"])

print("\nAll scenarios (2026-2035):")
print(df_deficit_full[df_deficit_full.year.between(2026,2035)]
      .pivot(index="year", columns="scenario", values="deficit_billions").to_string())

print("\n10yr totals:")
for sc in df_deficit_full.scenario.unique():
    t = df_deficit_full[(df_deficit_full.scenario == sc) & df_deficit_full.year.between(2026,2035)]["deficit_billions"].sum()
    print(f"  {sc:30s}: ${t:,.0f}B")

df_deficit_full.to_csv(OUT_DIR / "projections_deficit.csv", index=False)
print(f"\nUpdated projections_deficit.csv ({len(df_deficit_full)} rows)")

Tariff technical change by year (billions):
  2026: 351.844
  2027: 352.512
  2028: 356.668
  2029: 354.58
  2030: 351.274
  2031: 345.856
  2032: 334.095
  2033: 330.486
  2034: 330.948
  2035: 340.472

All scenarios (2026-2035):
scenario  feb_2026_current_law  jan_2025_baseline  no_tariff_revenue
year                                                                
2026                    1852.7             1713.3             2204.5
2027                    1887.2             1686.9             2239.7
2028                    2079.7             1910.8             2436.4
2029                    2019.7             1938.3             2374.3
2030                    2200.6             2139.7             2551.9
2031                    2285.7             2232.8             2631.6
2032                    2439.4             2370.6             2773.5
2033                    2780.7             2637.3             3111.2
2034                    2818.5             2597.2             3149.4
2035      

In [51]:
df_deficit_full.to_csv(OUT_DIR / "projections_deficit.csv", index=False)
print(f"\nUpdated projections_deficit.csv ({len(df_deficit_full)} rows)")


Updated projections_deficit.csv (32 rows)


In [52]:
"""
Cells 13–18 for cbo_projections.ipynb
Parses CBO pub. 61570 (OBBBA cost estimate) to extract TCJA-specific revenue costs
and constructs the 'tcja_extended_no_obbba' (Scenario 2).

FILE: 61570pl119212025ReconCLB.xlsx
- All amounts in MILLIONS of dollars (not billions)
- Header row 11: years in cols 4-13 (2025-2034), 5-yr total in col14, 10-yr in col15
- 'Estimated Revenues' rows use POSITIVE = revenue gain, NEGATIVE = revenue loss
  (i.e. negative means the provision COSTS money / reduces revenues)
- Deficit impact: revenue loss -> positive deficit effect

Data already in memory from prior cells:
  jan_def   = dict {year: deficit_billions}  (jan_2025_baseline, 2026-2035)
  jan_ni    = dict {year: net_interest_billions}
  feb_def   = dict {year: deficit_billions}  (feb_2026_current_law)
  feb_ni    = dict {year: net_interest_billions}
"""

# ─── CELL 13: Peek at 61570 structure ─────────────────────────────────────────

FILE_61570 = "61570-pl119-21-2025Recon-CLB.xlsx"

import openpyxl

wb_61570 = openpyxl.load_workbook(FILE_61570, data_only=True)
print("Sheets:", wb_61570.sheetnames)
ws7 = wb_61570["Title VII"]
print(f"Title VII: {ws7.max_row} rows × {ws7.max_column} cols")

# Header sanity check
print("Year headers (row 11, cols 4-15):",
      [ws7.cell(11, c).value for c in range(4, 16)])




Sheets: ['Summary Table', 'Title I', 'Title II', 'Title III', 'Title IV', 'Title V', 'Title VI', 'Title VII', 'Title VIII', 'Title IX', 'Title X', 'Interactions']
Title VII: 973 rows × 35 cols
Year headers (row 11, cols 4-15): [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, '2025-\n2029', '2025-\n2034']


In [53]:
# ─── CELL 14: Helper + column map for 61570 ───────────────────────────────────

YEARS_61570 = list(range(2025, 2035))   # 2025-2034, cols 4-13

def get_row_61570(ws, row_idx):
    """Return {year: value_millions} for a data row (cols 4-13). '*' -> 0."""
    out = {}
    for i, yr in enumerate(YEARS_61570):
        v = ws.cell(row_idx, 4 + i).value
        if v is None or v == "*":
            v = 0
        out[yr] = float(v)
    return out


In [54]:


# ─── CELL 15: Parse Title VII — TCJA-specific revenue rows ───────────────────
#
# TCJA "extension" provisions are in Chapter 1 of Subtitle A Tax (rows 372-458).
# These are all the provisions that were originally enacted in 2017 and would have
# expired — making them the pure TCJA extension cost.
#
# We capture the following sections from Chapter 1 (Sec. 70101-70120):
#   Row 375: 70101 — Reduced Rates (individual income tax rates)
#   Row 379: 70102 — Standard Deduction
#   Row 383: 70103 — Termination of Personal Exemptions (offsets)
#   Row 387: 70104 — Child Tax Credit (CTC)
#   Row 391: 70105 — Qualified Business Income (pass-through, 20% deduction)
#   Row 395: 70106 — Estate & Gift Tax Exemption
#   Row 400: 70107 — AMT Exemption / Phase-out
#   Row 404: 70108 — Mortgage Interest Deduction cap
#   Row 408: 70109 — Casualty Loss Limitation
#   Row 412: 70110 — Misc. Itemized Deductions (suspension)
#   Row 415: 70111 — Itemized Deduction overall limitation (Pease repeal)
#   Row 419: 70112 — Transportation Fringe Benefits
#   Row 425: 70113 — Moving Expense Deduction
#   Row 431: 70114 — Wagering Loss Limitation
#   Row 435: 70115 — ABLE contributions
#   Row 439: 70116 — ABLE savers credit
#   Row 443: 70117 — 529→ABLE rollovers
#   Row 448: 70118 — Sinai Peninsula combat zone
#   Row 453: 70119 — Student loan discharge exclusion
#   Row 457: 70120 — SALT cap ($10k)
#
# NOTE: Row 172 "Subtotal, Subtitle A" (outlays) and Row 807 (revenues subtotal)
# would give the full Subtitle A total. We use the granular rows so we can isolate
# pure TCJA (Chapter 1) vs. new provisions (Chapter 2+).

TCJA_CHAPTER1_REV_ROWS = {
    "70101_rates":              375,
    "70102_std_deduction":      379,
    "70103_personal_exemptions":383,
    "70104_ctc":                387,
    "70105_qbi":                391,
    "70106_estate_gift":        395,
    "70107_amt":                400,
    "70108_mortgage_interest":  404,
    "70109_casualty_loss":      408,
    "70110_misc_itemized":      412,
    "70111_pease":              415,
    "70112_transportation":     419,
    "70113_moving_expense":     425,
    "70114_wagering":           431,
    "70115_able_contrib":       435,
    "70116_able_savers":        439,
    "70117_529_able":           443,
    "70118_sinai":              448,
    "70119_student_loan":       453,
    "70120_salt":               457,
}

tcja_ch1_by_provision = {}
for label, row_idx in TCJA_CHAPTER1_REV_ROWS.items():
    tcja_ch1_by_provision[label] = get_row_61570(ws7, row_idx)

# Sum across all provisions to get total TCJA Chapter 1 annual revenue change (millions)
tcja_ch1_annual_rev_M = {}
for yr in YEARS_61570:
    tcja_ch1_annual_rev_M[yr] = sum(
        tcja_ch1_by_provision[p][yr] for p in tcja_ch1_by_provision
    )

print("TCJA Chapter 1 — annual revenue change (millions):")
for yr in YEARS_61570:
    print(f"  {yr}: {tcja_ch1_annual_rev_M[yr]:>12,.0f}")

ten_yr = sum(tcja_ch1_annual_rev_M[yr] for yr in range(2025, 2035))
print(f"\n10-yr total (2025-2034): ${ten_yr/1e6:,.3f}T")
# Expected: roughly -$3.7 to -$3.9T (revenue reduction)
# CRFB pegs TCJA individual extension at ~$3.9T including interaction effects



TCJA Chapter 1 — annual revenue change (millions):
  2025:      -28,898
  2026:     -269,144
  2027:     -413,437
  2028:     -434,574
  2029:     -437,618
  2030:     -422,698
  2031:     -417,202
  2032:     -439,292
  2033:     -456,047
  2034:     -475,895

10-yr total (2025-2034): $-3.795T


In [55]:

# ─── CELL 16: Spot-check against Summary Table ───────────────────────────────

ws_sum = wb_61570["Summary Table"]
# Row 45: Title VII Estimated Revenues (total, all of Title VII)
t7_rev_total = get_row_61570(ws_sum, 45)
print("Title VII total revenue effect (millions):")
for yr in YEARS_61570:
    print(f"  {yr}: {t7_rev_total[yr]:>12,.0f}")

t7_10yr = sum(t7_rev_total[yr] for yr in range(2025, 2035))
print(f"\nTitle VII 10-yr revenue total: ${t7_10yr/1e6:,.3f}T")
# Should be around -$4.5 to -$4.9T (includes biz tax, tips/OT, IRA clawbacks)



Title VII total revenue effect (millions):
  2025:     -130,903
  2026:     -461,569
  2027:     -585,848
  2028:     -585,287
  2029:     -529,328
  2030:     -450,023
  2031:     -416,941
  2032:     -425,102
  2033:     -452,408
  2034:     -483,910

Title VII 10-yr revenue total: $-4.521T


In [56]:
import pandas as pd

# ── Reconstruct scenario dicts from CSVs ──────────────────────────────────────
df_def = pd.read_csv("../public/projections_deficit.csv")
df_ni  = pd.read_csv("../public/projections_net_interest.csv")

jan_def = df_def[df_def["scenario"] == "jan_2025_baseline"].set_index("year")["deficit_billions"].to_dict()
feb_def = df_def[df_def["scenario"] == "feb_2026_current_law"].set_index("year")["deficit_billions"].to_dict()
jan_ni  = df_ni[df_ni["scenario"] == "jan_2025_baseline"].set_index("year")["net_interest_billions"].to_dict()
feb_ni  = df_ni[df_ni["scenario"] == "feb_2026_current_law"].set_index("year")["net_interest_billions"].to_dict()

PROJ_YEARS = list(range(2026, 2036))  # 2026-2035

# ── TCJA deficit addition (billions) — from Cell 15 output ───────────────────
# Revenue change is negative (loss), so negate to get deficit increase.
# 2035 not in 61570 (file ends at 2034); hold 2034 value flat.
tcja_rev_M = {
    2026: -269_144, 2027: -413_437, 2028: -434_574, 2029: -437_618,
    2030: -422_698, 2031: -417_202, 2032: -439_292, 2033: -456_047,
    2034: -475_895, 2035: -475_895,   # 2035 extrapolated flat from 2034
}
tcja_def_add_B = {yr: -tcja_rev_M[yr] / 1000.0 for yr in PROJ_YEARS}

# ── Scenario 2 deficits ───────────────────────────────────────────────────────
tcja_no_obbba_def = {yr: jan_def[yr] + tcja_def_add_B[yr] for yr in PROJ_YEARS}

print("=" * 75)
print("SCENARIO 2 DEFICITS (billions)")
print("=" * 75)
print(f"{'Year':<6} {'Jan Baseline':>14} {'+ TCJA Add':>12} {'= Scenario 2':>14}")
print("-" * 50)
for yr in PROJ_YEARS:
    print(f"{yr:<6} {jan_def[yr]:>14.1f} {tcja_def_add_B[yr]:>12.1f} {tcja_no_obbba_def[yr]:>14.1f}")
s2_def_10yr = sum(tcja_no_obbba_def[yr] for yr in PROJ_YEARS)
jan_def_10yr = sum(jan_def[yr] for yr in PROJ_YEARS)
print(f"{'10yr':6} {jan_def_10yr:>14.1f} {sum(tcja_def_add_B[yr] for yr in PROJ_YEARS):>12.1f} {s2_def_10yr:>14.1f}")
print(f"       = ${jan_def_10yr/1000:.2f}T              = ${s2_def_10yr/1000:.2f}T")


# ── NET INTEREST: Three modeling approaches ───────────────────────────────────
print("\n" + "=" * 75)
print("NET INTEREST MODELING APPROACHES")
print("=" * 75)

# ── Approach A: Proportional to deficit (Jan baseline NI/deficit ratio) ───────
# Most conservative. Says: Scenario 2 carries same NI-to-deficit ratio as Jan baseline.
# Rationale: Jan baseline already reflects market expectations of a TCJA-extended world
# (since TCJA extension was widely anticipated). So its NI curve is a reasonable anchor.
ni_A = {}
for yr in PROJ_YEARS:
    ratio = jan_ni[yr] / jan_def[yr]
    ni_A[yr] = tcja_no_obbba_def[yr] * ratio

# ── Approach B: Proportional scaling off feb baseline ─────────────────────────
# Says: the incremental NI per dollar of incremental deficit equals the feb/jan ratio.
# Problem: feb-jan deficit gap is small/negative in some years (tariff offsets dominate),
# causing the ratio to blow up. Shown here for completeness — expect instability.
ni_B = {}
for yr in PROJ_YEARS:
    def_gap = feb_def.get(yr, 0) - jan_def[yr]
    ni_gap  = feb_ni.get(yr, 0) - jan_ni[yr]
    ratio   = (ni_gap / def_gap) if abs(def_gap) > 10 else (jan_ni[yr] / jan_def[yr])
    ni_B[yr] = jan_ni[yr] + tcja_def_add_B[yr] * ratio

# ── Approach C: Additive fixed NI premium (simple, transparent) ───────────────
# Uses a fixed ~3.5% cost-of-debt applied to the cumulative additional deficit stock.
# Approximates: each year's TCJA addition accumulates, and we pay ~3.5% on that stock.
# This is rough but mechanically honest about how interest compounds.
COST_OF_DEBT = 0.035
cumulative_extra_debt = 0.0
ni_C = {}
for yr in PROJ_YEARS:
    cumulative_extra_debt += tcja_def_add_B[yr]
    ni_C[yr] = jan_ni[yr] + cumulative_extra_debt * COST_OF_DEBT

# ── Print comparison ──────────────────────────────────────────────────────────
print(f"\n{'Year':<6} {'Jan NI':>9} {'A: Prop Jan':>12} {'B: Prop Feb':>12} {'C: Additive':>12}")
print(f"       {'':>9} {'(conservative)':>12} {'(unstable?)':>12} {'(3.5% debt)':>12}")
print("-" * 55)
for yr in PROJ_YEARS:
    print(f"{yr:<6} {jan_ni[yr]:>9.1f} {ni_A[yr]:>12.1f} {ni_B[yr]:>12.1f} {ni_C[yr]:>12.1f}")
print("-" * 55)
print(f"{'10yr':6} {sum(jan_ni[yr] for yr in PROJ_YEARS):>9.1f} "
      f"{sum(ni_A[yr] for yr in PROJ_YEARS):>12.1f} "
      f"{sum(ni_B[yr] for yr in PROJ_YEARS):>12.1f} "
      f"{sum(ni_C[yr] for yr in PROJ_YEARS):>12.1f}")
print(f"\n       Jan NI 10yr = ${sum(jan_ni[yr] for yr in PROJ_YEARS)/1000:.2f}T")
print(f"       A 10yr       = ${sum(ni_A[yr] for yr in PROJ_YEARS)/1000:.2f}T")
print(f"       B 10yr       = ${sum(ni_B[yr] for yr in PROJ_YEARS)/1000:.2f}T")
print(f"       C 10yr       = ${sum(ni_C[yr] for yr in PROJ_YEARS)/1000:.2f}T")

# ── Sanity checks ─────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("SANITY CHECKS")
print("=" * 75)
print("\nFor each approach, Scenario 2 NI should be:")
print("  (a) > Jan baseline NI  (more debt = more interest)")
print("  (b) Monotonically reasonable (no wild jumps)")
print("  (c) < No-tariff scenario NI  (Scenario 2 deficit < no-tariff deficit)")
print()

no_tar_def = df_def[df_def["scenario"] == "no_tariff_revenue"].set_index("year")["deficit_billions"].to_dict()

for label, ni_dict in [("A (prop jan)", ni_A), ("B (prop feb)", ni_B), ("C (additive)", ni_C)]:
    all_above_jan = all(ni_dict[yr] >= jan_ni[yr] for yr in PROJ_YEARS)
    max_yoy = max(abs(ni_dict[yr] - ni_dict[yr-1]) for yr in PROJ_YEARS[1:])
    print(f"  {label}: above_jan={all_above_jan}  max_YoY_jump=${max_yoy:.0f}B")

SCENARIO 2 DEFICITS (billions)
Year     Jan Baseline   + TCJA Add   = Scenario 2
--------------------------------------------------
2026           1713.3        269.1         1982.4
2027           1686.9        413.4         2100.3
2028           1910.8        434.6         2345.4
2029           1938.3        437.6         2375.9
2030           2139.7        422.7         2562.4
2031           2232.8        417.2         2650.0
2032           2370.6        439.3         2809.9
2033           2637.3        456.0         3093.3
2034           2597.2        475.9         3073.1
2035           2531.4        475.9         3007.3
10yr          21758.3       4241.8        26000.1
       = $21.76T              = $26.00T

NET INTEREST MODELING APPROACHES

Year      Jan NI  A: Prop Jan  B: Prop Feb  C: Additive
                 (conservative)  (unstable?)  (3.5% debt)
-------------------------------------------------------
2026      1010.3       1169.0       1065.7       1019.7
2027      1075.2 

In [57]:
 
# %% Cell 18 — Write Scenario 2 (tcja_extended_no_obbba) to CSVs and validate
# ni_A and tcja_no_obbba_def are in scope from Cell 17 (the modeling cell)
 
import pandas as pd, os
 
OUTPUT_DIR   = "../public/"
def_csv_path = os.path.join(OUTPUT_DIR, "projections_deficit.csv")
ni_csv_path  = os.path.join(OUTPUT_DIR, "projections_net_interest.csv")
 
df_def = pd.read_csv(def_csv_path)
df_ni  = pd.read_csv(ni_csv_path)
 
# Idempotent — drop any prior Scenario 2 rows
df_def = df_def[df_def["scenario"] != "tcja_extended_no_obbba"]
df_ni  = df_ni[df_ni["scenario"]  != "tcja_extended_no_obbba"]
 
# Build Scenario 2 rows (deficit + NI using Approach A)
new_def_rows = pd.DataFrame([
    {"scenario": "tcja_extended_no_obbba", "year": yr, "deficit_billions": round(tcja_no_obbba_def[yr], 3)}
    for yr in PROJ_YEARS
])
new_ni_rows = pd.DataFrame([
    {"scenario": "tcja_extended_no_obbba", "year": yr, "net_interest_billions": round(ni_A[yr], 3)}
    for yr in PROJ_YEARS
])
 
df_def_out = pd.concat([df_def, new_def_rows], ignore_index=True).sort_values(["scenario", "year"])
df_ni_out  = pd.concat([df_ni,  new_ni_rows],  ignore_index=True).sort_values(["scenario", "year"])
 
df_def_out.to_csv(def_csv_path, index=False)
df_ni_out.to_csv(ni_csv_path,   index=False)
print("✓ projections_deficit.csv written")
print("✓ projections_net_interest.csv written")
 
# ── Final validation ──────────────────────────────────────────────────────────
print("\n=== ALL SCENARIOS — DEFICIT (billions) ===")
pivot_def = df_def_out.pivot(index="year", columns="scenario", values="deficit_billions")
print(pivot_def.to_string())
print("\n10yr totals (2026-2035):")
for col in pivot_def.columns:
    t = df_def_out[(df_def_out["scenario"]==col) & df_def_out["year"].between(2026,2035)]["deficit_billions"].sum()
    print(f"  {col:40s}: ${t/1000:.2f}T")
 
print("\n=== ALL SCENARIOS — NET INTEREST (billions) ===")
pivot_ni = df_ni_out.pivot(index="year", columns="scenario", values="net_interest_billions")
print(pivot_ni.to_string())
print("\n10yr totals (2026-2035):")
for col in pivot_ni.columns:
    t = df_ni_out[(df_ni_out["scenario"]==col) & df_ni_out["year"].between(2026,2035)]["net_interest_billions"].sum()
    print(f"  {col:40s}: ${t/1000:.2f}T")
 
# ── Ordering checks ───────────────────────────────────────────────────────────
print("\n=== ORDERING CHECKS ===")
print(f"{'Year':<6} {'jan':>8} {'tcja':>8} {'feb':>8} {'notar':>8}  {'def ok?':>8}  {'ni ok?':>8}")
print("-" * 65)
errors = []
for yr in range(2026, 2036):
    def_row = {sc: df_def_out[(df_def_out["scenario"]==sc) & (df_def_out["year"]==yr)]["deficit_billions"].values
               for sc in ["jan_2025_baseline","tcja_extended_no_obbba","feb_2026_current_law","no_tariff_revenue"]}
    ni_row  = {sc: df_ni_out[(df_ni_out["scenario"]==sc) & (df_ni_out["year"]==yr)]["net_interest_billions"].values
               for sc in ["jan_2025_baseline","tcja_extended_no_obbba","feb_2026_current_law","no_tariff_revenue"]}
    if all(len(v) for v in def_row.values()) and all(len(v) for v in ni_row.values()):
        jan_d, s2_d, feb_d, not_d = [def_row[s][0] for s in ["jan_2025_baseline","tcja_extended_no_obbba","feb_2026_current_law","no_tariff_revenue"]]
        jan_n, s2_n, feb_n, not_n = [ni_row[s][0]  for s in ["jan_2025_baseline","tcja_extended_no_obbba","feb_2026_current_law","no_tariff_revenue"]]
        def_ok = "✓" if jan_d < s2_d else "✗"
        ni_ok  = "✓" if jan_n < s2_n else "✗"
        print(f"{yr:<6} {jan_d:>8.0f} {s2_d:>8.0f} {feb_d:>8.0f} {not_d:>8.0f}  {def_ok:>8}  {ni_ok:>8}")
        if def_ok == "✗" or ni_ok == "✗":
            errors.append(yr)
 
if errors:
    print(f"\n✗ Issues in years: {errors}")
else:
    print("\n✓ All ordering checks pass")
 

✓ projections_deficit.csv written
✓ projections_net_interest.csv written

=== ALL SCENARIOS — DEFICIT (billions) ===
scenario  feb_2026_current_law  jan_2025_baseline  no_tariff_revenue  tcja_extended_no_obbba
year                                                                                        
2025                    1775.4                NaN                NaN                     NaN
2026                    1852.7             1713.3             2204.5                1982.444
2027                    1887.2             1686.9             2239.7                2100.337
2028                    2079.7             1910.8             2436.4                2345.374
2029                    2019.7             1938.3             2374.3                2375.918
2030                    2200.6             2139.7             2551.9                2562.398
2031                    2285.7             2232.8             2631.6                2650.002
2032                    2439.4             237